# CDC PLACES - Health Data Exploratory Analysis

**Data Sources:**
- `cdc_places`: 12,914 rows - Raw health measure data
- `cdc_places_wide`: 1,174 rows - Pivoted format (one row per geography)
- `cdc_places_zcta`: 12,914 rows - ZCTA-specific health measures

**Objectives:**
1. Understand health measure coverage across NJ
2. Identify geographic health disparities
3. Analyze correlations between health measures
4. Compare county and ZCTA-level patterns
5. Identify areas of concern for public health interventions

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Connect to database
db_path = Path('../data/db/nj_pipeline.duckdb')
conn = duckdb.connect(str(db_path), read_only=True)
print(f"Connected to: {db_path}")

Connected to: ../data/db/nj_pipeline.duckdb


## 1. Data Overview

In [2]:
# Schema information
print("=" * 60)
print("CDC PLACES WIDE SCHEMA")
print("=" * 60)
display(conn.execute("DESCRIBE cdc_places_wide").df())

,column_name,column_type,null,key,default,extra
0,zcta,VARCHAR,YES,None,None,None
1,location_name,VARCHAR,YES,None,None,None
2,year,BIGINT,YES,None,None,None
3,access2,DOUBLE,YES,None,None,None
4,arthritis,DOUBLE,YES,None,None,None
5,bphigh,DOUBLE,YES,None,None,None
6,cancer,DOUBLE,YES,None,None,None
7,casthma,DOUBLE,YES,None,None,None
8,chd,DOUBLE,YES,None,None,None
9,checkup,DOUBLE,YES,None,None,None


In [3]:
# Sample data
print("\nSample Data (first 3 records):")
sample = conn.execute("""
    SELECT * FROM cdc_places_wide LIMIT 3
""").df()
display(sample.T)  # Transpose to see all columns

,0,1,2
zcta,07001,07001,07002
location_name,07001,07001,07002
year,2022,2023,2022
access2,NaN,12.5,NaN
arthritis,NaN,18.4,NaN
bphigh,NaN,31.8,NaN
cancer,NaN,5.1,NaN
casthma,NaN,8.7,NaN
chd,NaN,4.5,NaN
checkup,NaN,76.6,NaN


In [5]:
# Data coverage
coverage = conn.execute("""
    SELECT 
        geo_type,
        COUNT(*) as num_records,
        COUNT(DISTINCT location_name) as num_locations,
        MIN(year) as earliest_year,
        MAX(year) as latest_year
    FROM cdc_places_wide
    GROUP BY geo_type
""").df()

print("\nData Coverage:")
display(coverage)

BinderException: Binder Error: Referenced column "geo_type" not found in FROM clause!
Candidate bindings: "obesity", "diabetes", "depression", "sleep", "ghlth"

LINE 9:     GROUP BY geo_type
                     ^

## 2. Available Health Measures

In [6]:
# Get list of health measure columns
all_cols = conn.execute("SELECT * FROM cdc_places_wide LIMIT 0").df().columns.tolist()
id_cols = ['location_id', 'location_name', 'geo_type', 'state_abbr', 'county_name', 
           'county_fips', 'year', 'total_population']
health_measures = [col for col in all_cols if col not in id_cols]

print(f"\nTotal health measures available: {len(health_measures)}")
print("\nHealth Measures:")
for i, measure in enumerate(sorted(health_measures), 1):
    print(f"  {i:2d}. {measure}")


Total health measures available: 23

Health Measures:
   1. access2
   2. arthritis
   3. bphigh
   4. cancer
   5. casthma
   6. chd
   7. checkup
   8. cholscreen
   9. copd
  10. csmoking
  11. dental
  12. depression
  13. diabetes
  14. ghlth
  15. highchol
  16. lpa
  17. mhlth
  18. obesity
  19. phlth
  20. sleep
  21. stroke
  22. teethlost
  23. zcta


In [7]:
# Data completeness by measure
completeness_query = """
    SELECT 
        COUNT(*) as total_records,
        {} as measure_name,
        COUNT({}) as non_null_count,
        ROUND(COUNT({}) * 100.0 / COUNT(*), 1) as pct_complete
    FROM cdc_places_wide
"""

completeness_data = []
for measure in health_measures[:20]:  # Check first 20 measures
    result = conn.execute(completeness_query.format(f"'{measure}'", measure, measure)).fetchone()
    completeness_data.append({
        'measure': measure,
        'non_null_count': result[2],
        'pct_complete': result[3]
    })

completeness_df = pd.DataFrame(completeness_data).sort_values('pct_complete', ascending=False)
print("\nData Completeness (Top 20 Measures):")
display(completeness_df.head(20))

,measure,non_null_count,pct_complete
0,zcta,1174,100.0
1,access2,587,50.0
18,obesity,587,50.0
17,mhlth,587,50.0
16,lpa,587,50.0
15,highchol,587,50.0
14,ghlth,587,50.0
13,diabetes,587,50.0
12,depression,587,50.0
11,dental,587,50.0


## 3. ZCTA-Level Health Analysis

In [8]:
# Get ZCTA data
zcta_data = conn.execute("""
    SELECT *
    FROM cdc_places_wide
    WHERE geo_type = 'ZCTA'
    ORDER BY location_name
""").df()

print(f"\nZCTA-Level Data: {len(zcta_data)} zip codes")
print(f"Year(s): {zcta_data['year'].unique()}")
print(f"\nSample of first 5 ZCTAs:")
display(zcta_data[['location_name', 'county_name', 'total_population']].head())

BinderException: Binder Error: Referenced column "geo_type" not found in FROM clause!
Candidate bindings: "obesity", "diabetes", "depression", "sleep", "ghlth"

LINE 4:     WHERE geo_type = 'ZCTA'
                  ^

In [ ]:
# Focus on key chronic disease measures
key_measures = {
    'obesity': 'Obesity',
    'diabetes': 'Diabetes', 
    'bphigh': 'High Blood Pressure',
    'chd': 'Coronary Heart Disease',
    'csmoking': 'Current Smoking',
    'depression': 'Depression',
    'mhlth': 'Mental Health Not Good',
    'lpa': 'No Leisure Physical Activity'
}

# Check which measures are available
available_measures = {k: v for k, v in key_measures.items() if k in zcta_data.columns}
print(f"\nAvailable key measures: {len(available_measures)}")
for code, name in available_measures.items():
    print(f"  • {code}: {name}")

In [ ]:
# Summary statistics for key measures
if available_measures:
    summary_stats = zcta_data[list(available_measures.keys())].describe().T
    summary_stats['measure_name'] = summary_stats.index.map(available_measures)
    summary_stats = summary_stats[['measure_name', 'count', 'mean', '50%', 'min', 'max']]
    summary_stats.columns = ['Measure', 'N', 'Mean %', 'Median %', 'Min %', 'Max %']
    
    print("\nHealth Measure Statistics Across NJ ZCTAs:")
    display(summary_stats.round(1))

In [ ]:
# Distribution plots for key measures
if available_measures and len(available_measures) >= 4:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for idx, (measure_code, measure_name) in enumerate(list(available_measures.items())[:8]):
        if measure_code in zcta_data.columns:
            data = zcta_data[measure_code].dropna()
            axes[idx].hist(data, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
            axes[idx].axvline(data.median(), color='red', linestyle='--', 
                             label=f'Median: {data.median():.1f}%')
            axes[idx].set_xlabel('Prevalence (%)')
            axes[idx].set_ylabel('Number of ZCTAs')
            axes[idx].set_title(measure_name)
            axes[idx].legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

## 4. Geographic Health Disparities

In [ ]:
# Best and worst performing ZCTAs for each measure
if available_measures:
    print("\nHealthiest ZCTAs (Lowest Prevalence):")
    print("=" * 80)
    
    for measure_code, measure_name in list(available_measures.items())[:4]:
        if measure_code in zcta_data.columns:
            best = zcta_data.nsmallest(5, measure_code)[['location_name', 'county_name', measure_code]]
            print(f"\n{measure_name}:")
            for _, row in best.iterrows():
                print(f"  {row['location_name']} ({row['county_name']}): {row[measure_code]:.1f}%")

In [ ]:
if available_measures:
    print("\nZCTAs with Highest Health Burden (Highest Prevalence):")
    print("=" * 80)
    
    for measure_code, measure_name in list(available_measures.items())[:4]:
        if measure_code in zcta_data.columns:
            worst = zcta_data.nlargest(5, measure_code)[['location_name', 'county_name', measure_code]]
            print(f"\n{measure_name}:")
            for _, row in worst.iterrows():
                print(f"  {row['location_name']} ({row['county_name']}): {row[measure_code]:.1f}%")

## 5. County-Level Comparisons

In [ ]:
# County-level aggregates
if available_measures and 'county_name' in zcta_data.columns:
    county_health = zcta_data.groupby('county_name')[list(available_measures.keys())].agg(['mean', 'median']).round(1)
    
    # Show obesity rates by county (example)
    if 'obesity' in available_measures:
        obesity_by_county = zcta_data.groupby('county_name')['obesity'].agg(['mean', 'count']).sort_values('mean', ascending=False)
        obesity_by_county.columns = ['Avg Obesity %', 'Num ZCTAs']
        
        print("\nObesity Prevalence by County:")
        display(obesity_by_county)
        
        # Plot
        fig, ax = plt.subplots(figsize=(12, 8))
        obesity_by_county_sorted = obesity_by_county.sort_values('Avg Obesity %')
        ax.barh(obesity_by_county_sorted.index, obesity_by_county_sorted['Avg Obesity %'], alpha=0.7)
        ax.set_xlabel('Average Obesity Prevalence (%)')
        ax.set_title('Obesity Prevalence by County (Average Across ZCTAs)')
        ax.grid(axis='x', alpha=0.3)
        
        for i, (county, value) in enumerate(zip(obesity_by_county_sorted.index, obesity_by_county_sorted['Avg Obesity %'])):
            ax.text(value, i, f' {value:.1f}%', va='center', fontsize=9)
        
        plt.tight_layout()
        plt.show()

## 6. Correlation Analysis

In [ ]:
# Correlation matrix for health measures
if available_measures and len(available_measures) >= 4:
    measure_cols = list(available_measures.keys())
    corr_matrix = zcta_data[measure_cols].corr()
    
    # Plot correlation heatmap
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8},
                xticklabels=[available_measures[col] for col in measure_cols],
                yticklabels=[available_measures[col] for col in measure_cols])
    ax.set_title('Correlation Between Health Measures', fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()
    
    # Find strongest correlations
    print("\nStrongest Positive Correlations:")
    corr_pairs = []
    for i in range(len(measure_cols)):
        for j in range(i+1, len(measure_cols)):
            corr_pairs.append({
                'measure1': available_measures[measure_cols[i]],
                'measure2': available_measures[measure_cols[j]],
                'correlation': corr_matrix.iloc[i, j]
            })
    
    corr_df = pd.DataFrame(corr_pairs).sort_values('correlation', ascending=False)
    display(corr_df.head(10))

## 7. Composite Health Index

In [ ]:
# Create a simple composite health index (average of key measures)
if available_measures and len(available_measures) >= 4:
    # Select measures for composite (higher = worse health)
    composite_measures = ['obesity', 'diabetes', 'bphigh', 'csmoking', 'depression']
    composite_measures = [m for m in composite_measures if m in available_measures]
    
    if composite_measures:
        zcta_data['health_burden_index'] = zcta_data[composite_measures].mean(axis=1)
        
        print(f"\nComposite Health Burden Index")
        print(f"Based on: {', '.join([available_measures[m] for m in composite_measures])}")
        print(f"\nIndex Statistics:")
        print(zcta_data['health_burden_index'].describe())
        
        # Top and bottom ZCTAs
        print("\nHealthiest ZCTAs (Lowest Burden):")
        best_health = zcta_data.nsmallest(10, 'health_burden_index')[['location_name', 'county_name', 'health_burden_index']]
        display(best_health)
        
        print("\nZCTAs with Highest Health Burden:")
        worst_health = zcta_data.nlargest(10, 'health_burden_index')[['location_name', 'county_name', 'health_burden_index']]
        display(worst_health)
        
        # Distribution plot
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.hist(zcta_data['health_burden_index'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='coral')
        ax.axvline(zcta_data['health_burden_index'].median(), color='red', linestyle='--', 
                   label=f'Median: {zcta_data["health_burden_index"].median():.1f}')
        ax.set_xlabel('Health Burden Index')
        ax.set_ylabel('Number of ZCTAs')
        ax.set_title('Distribution of Composite Health Burden Index')
        ax.legend()
        plt.tight_layout()
        plt.show()

## 8. Population-Weighted Analysis

In [ ]:
# Calculate population-weighted prevalence
if available_measures and 'total_population' in zcta_data.columns:
    total_pop = zcta_data['total_population'].sum()
    
    weighted_prevalence = {}
    for measure_code, measure_name in available_measures.items():
        if measure_code in zcta_data.columns:
            weighted_avg = (zcta_data[measure_code] * zcta_data['total_population']).sum() / total_pop
            simple_avg = zcta_data[measure_code].mean()
            weighted_prevalence[measure_name] = {
                'weighted_avg': weighted_avg,
                'simple_avg': simple_avg,
                'difference': weighted_avg - simple_avg
            }
    
    weighted_df = pd.DataFrame(weighted_prevalence).T
    weighted_df = weighted_df.round(2)
    
    print(f"\nPopulation-Weighted vs Simple Average Prevalence:")
    print(f"Total NJ Population Covered: {total_pop:,}")
    print("\n")
    display(weighted_df.sort_values('weighted_avg', ascending=False))

## 9. Access to Care Measures

In [ ]:
# Check for healthcare access measures
access_measures = {
    'access2': 'Lack of Health Insurance',
    'checkup': 'Annual Checkup',
    'dental': 'Dental Visit',
    'mammouse': 'Mammography Use'
}

available_access = {k: v for k, v in access_measures.items() if k in zcta_data.columns}

if available_access:
    print("\nHealthcare Access Measures:")
    for code, name in available_access.items():
        median_val = zcta_data[code].median()
        print(f"  • {name}: {median_val:.1f}% (median)")
    
    # Plot access measures by county
    if 'access2' in available_access and 'county_name' in zcta_data.columns:
        access_by_county = zcta_data.groupby('county_name')['access2'].mean().sort_values(ascending=False)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.barh(access_by_county.index, access_by_county.values, alpha=0.7, color='indianred')
        ax.set_xlabel('Avg % Without Health Insurance')
        ax.set_title('Lack of Health Insurance by County')
        ax.grid(axis='x', alpha=0.3)
        
        for i, (county, value) in enumerate(zip(access_by_county.index, access_by_county.values)):
            ax.text(value, i, f' {value:.1f}%', va='center', fontsize=9)
        
        plt.tight_layout()
        plt.show()

## 10. Key Findings Summary

In [ ]:
print("="*70)
print("KEY FINDINGS - CDC PLACES HEALTH DATA")
print("="*70)

print(f"\n1. COVERAGE")
print(f"   • {len(zcta_data)} ZCTAs with health data")
print(f"   • {len(available_measures)} key health measures available")
print(f"   • Total population: {zcta_data['total_population'].sum():,}")

if available_measures:
    print(f"\n2. PREVALENCE (Median % across ZCTAs)")
    for code, name in list(available_measures.items())[:6]:
        if code in zcta_data.columns:
            median_val = zcta_data[code].median()
            print(f"   • {name}: {median_val:.1f}%")

if 'health_burden_index' in zcta_data.columns:
    print(f"\n3. HEALTH BURDEN INDEX")
    print(f"   • Median index: {zcta_data['health_burden_index'].median():.1f}")
    print(f"   • Range: {zcta_data['health_burden_index'].min():.1f} - {zcta_data['health_burden_index'].max():.1f}")
    best = zcta_data.nsmallest(1, 'health_burden_index').iloc[0]
    worst = zcta_data.nlargest(1, 'health_burden_index').iloc[0]
    print(f"   • Healthiest ZCTA: {best['location_name']} ({best['health_burden_index']:.1f})")
    print(f"   • Highest burden: {worst['location_name']} ({worst['health_burden_index']:.1f})")

if 'county_name' in zcta_data.columns and 'obesity' in zcta_data.columns:
    county_stats = zcta_data.groupby('county_name')['obesity'].mean().sort_values()
    print(f"\n4. COUNTY VARIATIONS (Obesity)")
    print(f"   • Lowest: {county_stats.index[0]} ({county_stats.iloc[0]:.1f}%)")
    print(f"   • Highest: {county_stats.index[-1]} ({county_stats.iloc[-1]:.1f}%)")

print("\n" + "="*70)

In [ ]:
conn.close()
print("\nAnalysis complete. Database connection closed.")